In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q -U unsloth "unsloth_zoo"
!pip install -q -U "transformers>=4.46" "trl>=0.12" peft accelerate bitsandbytes datasets python-Levenshtein

In [3]:
!pip install -q -U unsloth "unsloth_zoo"
!pip install -q datasets python-Levenshtein

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 25.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 17.2 MB/s eta 0:00:00
   ━

In [2]:
# ================= B1: Qwen2-VL-2B LoRA for receipt field extraction (CORD-v2) =================
import json, random, re
from datasets import load_dataset
from unsloth import FastVisionModel, is_bf16_supported

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
    load_in_4bit=True, use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,     # extraction: language layers carry the output format
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

ds = load_dataset("naver-clova-ix/cord-v2")

def flatten_cord(gt_str):
    try: parse = json.loads(gt_str).get("gt_parse", {})
    except Exception: return {}
    out = {}; menu = parse.get("menu", [])
    if isinstance(menu, dict): menu = [menu]
    items = [{"name": str(m.get("nm","")), "price": str(m.get("price",""))}
             for m in menu if isinstance(m, dict) and "nm" in m]
    if items: out["items"] = items
    total = parse.get("total", {})
    if isinstance(total, dict) and "total_price" in total: out["total"] = str(total["total_price"])
    return out

INSTR = ("Extract the receipt into JSON with keys: items (a list of name and price) "
         "and total. Output only JSON.")

def build_split(hf_split, cap=None):
    conv = []
    for ex in hf_split:
        tgt = flatten_cord(ex["ground_truth"])
        if not tgt: continue
        img = ex["image"].convert("RGB")
        if max(img.size) > 1024: img.thumbnail((1024,1024))
        conv.append({"messages":[
            {"role":"user","content":[{"type":"image","image":img},{"type":"text","text":INSTR}]},
            {"role":"assistant","content":[{"type":"text","text":json.dumps(tgt, ensure_ascii=False)}]},
        ]})
        if cap and len(conv) >= cap: break
    return conv

train_conv = build_split(ds["train"], cap=400)
val_raw    = [ex for ex in ds["validation"]][:60]
print("train:", len(train_conv), "| val:", len(val_raw))

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=2, learning_rate=2e-4,
        fp16=not is_bf16_supported(), bf16=is_bf16_supported(),
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs",
        report_to="none", remove_unused_columns=False,
        dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)
trainer.train()
model.save_pretrained("/kaggle/working/qwen2vl_cord_lora")
tokenizer.save_pretrained("/kaggle/working/qwen2vl_cord_lora")

# ---- eval: JSON validity + total exact-match + total edit ratio ----
try:
    import Levenshtein
    def edit_ratio(a,b): return 1 - Levenshtein.distance(a,b)/max(len(a),len(b),1) if (a or b) else 1.0
except Exception:
    def edit_ratio(a,b): return 1.0 if a==b else 0.0

def infer(pil_img):
    FastVisionModel.for_inference(model)
    msg = [{"role":"user","content":[{"type":"image","image":pil_img},{"type":"text","text":INSTR}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True,
                                        return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inp, max_new_tokens=256, do_sample=False)
    return tokenizer.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

valid_n=total_exact=n=0; edit_sum=0.0
for ex in val_raw:
    gold = flatten_cord(ex["ground_truth"])
    if not gold: continue
    img = ex["image"].convert("RGB")
    if max(img.size) > 1024: img.thumbnail((1024,1024))
    raw = infer(img); m = re.search(r"\{.*\}", raw, re.S); pred={}; ok=False
    if m:
        try: pred = json.loads(m.group(0)); ok=True
        except Exception: ok=False
    valid_n += int(ok)
    te = str(gold.get("total","")); pe = str(pred.get("total",""))
    total_exact += int(te==pe and te!=""); edit_sum += edit_ratio(pe, te); n += 1
print(f"n={n} | json_valid={valid_n/n:.3f} | total_exact={total_exact/n:.3f} | total_edit={edit_sum/n:.3f}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-b4aaeceff1d90e(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 400 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 2,227,450,368 (0.83% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.016028
20,0.175362
30,0.087198
40,0.054136
50,0.040283
60,0.026947
70,0.047084
80,0.033262
90,0.027816
100,0.030561


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen2vl_cord_lora/tokenizer_config.json.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_l

n=60 | json_valid=1.000 | total_exact=0.900 | total_edit=0.957


In [3]:
import random
for ex in random.sample(val_raw, 4):
    gold = flatten_cord(ex["ground_truth"])
    if not gold: continue
    img = ex["image"].convert("RGB")
    if max(img.size) > 1024: img.thumbnail((1024,1024))
    print("PRED:", infer(img))
    print("GOLD:", json.dumps(gold, ensure_ascii=False))
    print("-"*70)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PRED: {"items": [{"name": "M-Caramel Black Tea", "price": "28,000"}], "total": "28,000"}
GOLD: {"items": [{"name": "M-Caramel Black Tea", "price": "28,000"}], "total": "28,000"}
----------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PRED: {"items": [{"name": "NASI MERAH/PUTIH", "price": "5.000"}, {"name": "SAYUR", "price": "8.000"}, {"name": "KERUPUK/SAMBEL", "price": "2.000"}, {"name": "AYAM", "price": "14.000"}, {"name": "MINUMAN KEMASAN/REFILL", "price": "6.000"}], "total": "Rp. 35.000"}
GOLD: {"items": [{"name": "NASI MERAH/PUTIH", "price": "5.000"}, {"name": "SAYUR", "price": "8.000"}, {"name": "KERUPUK/SAMBEL", "price": "2.000"}, {"name": "AYAM", "price": "14.000"}, {"name": "MINUMAN KEMASAN/REFILL", "price": "6.000"}], "total": "Rp. 35.000"}
----------------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PRED: {"items": [{"name": "TALAM UNGU", "price": "19,500"}], "total": "11,700"}
GOLD: {"items": [{"name": "TALAM UNGU", "price": "19,500"}, {"name": "MIKA KECIL", "price": "0"}], "total": "11,700"}
----------------------------------------------------------------------
PRED: {"items": [{"name": "Mango Lemon Tea", "price": "Rp29,090"}, {"name": "Sliders Set", "price": "Rp113,636"}, {"name": "Chicken Vege Rice Bowl", "price": "Rp86,363"}, {"name": "Discount BCA 15%", "price": "-Rp34,363"}], "total": "Rp224,908"}
GOLD: {"items": [{"name": "Mango Lemon Tea", "price": "Rp 29,090"}, {"name": "Sliders Set", "price": "Rp 113,636"}, {"name": "Chicken Vege Rice Bowl", "price": "Rp 86,363"}, {"name": "Discount BCA 15%", "price": "-Rp 34,363"}], "total": "Rp 224,908"}
----------------------------------------------------------------------


In [4]:
!pip install -q sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 50.0 MB/s eta 0:00:0000:0100:01


In [5]:
# ---- RAG grounding layer (no retraining) ----
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

EXTRACTION_RULES = [
    "Preserve the currency symbol and spacing exactly as printed; if the receipt shows 'Rp 29,090' keep the space, do not output 'Rp29,090'.",
    "Include every line item, even ones with a price of 0.",
    "Use the same thousands separator printed on the receipt (comma or period); do not convert between them.",
    "The total is the final payable amount after any discounts.",
    "Copy item names verbatim, including abbreviations and slashes.",
    "Keep the minus sign on negative amounts such as discounts.",
]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
rule_emb = embedder.encode(EXTRACTION_RULES, normalize_embeddings=True).astype("float32")
rindex = faiss.IndexFlatIP(rule_emb.shape[1]); rindex.add(rule_emb)

def retrieve_rules(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = rindex.search(q, k)
    return [EXTRACTION_RULES[i] for i in idx[0]]

def grounded_infer(pil_img, query="receipt currency formatting spacing zero-price line items total"):
    FastVisionModel.for_inference(model)
    rules = retrieve_rules(query, k=3)
    instr = INSTR + "\nFollow these formatting rules:\n" + "\n".join(f"- {r}" for r in rules)
    msg = [{"role":"user","content":[{"type":"image","image":pil_img},{"type":"text","text":instr}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True,
                                        return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inp, max_new_tokens=256, do_sample=False)
    return tokenizer.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

# ---- measure whether RAG grounding actually helps (honest before/after) ----
import re, json
try:
    import Levenshtein
    def edit_ratio(a,b): return 1 - Levenshtein.distance(a,b)/max(len(a),len(b),1) if (a or b) else 1.0
except Exception:
    def edit_ratio(a,b): return 1.0 if a==b else 0.0

def parse_total(raw):
    m = re.search(r"\{.*\}", raw, re.S)
    if not m: return ""
    try: return str(json.loads(m.group(0)).get("total",""))
    except Exception: return ""

base_exact=grnd_exact=n=0
for ex in val_raw:
    gold = flatten_cord(ex["ground_truth"])
    if not gold: continue
    img = ex["image"].convert("RGB")
    if max(img.size) > 1024: img.thumbnail((1024,1024))
    g = str(gold.get("total",""))
    base_exact += int(parse_total(infer(img))==g and g!="")
    grnd_exact += int(parse_total(grounded_infer(img))==g and g!="")
    n += 1
print(f"total exact-match  |  base={base_exact/n:.3f}  RAG-grounded={grnd_exact/n:.3f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

total exact-match  |  base=0.900  RAG-grounded=0.883


In [4]:
# reload the fine-tuned adapter (no retraining) + data + helpers
import json, re
from datasets import load_dataset
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "/kaggle/working/qwen2vl_cord_lora",   # your saved adapter
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)

ds = load_dataset("naver-clova-ix/cord-v2")
val_raw = [ex for ex in ds["validation"]][:60]

def flatten_cord(gt_str):
    try: parse = json.loads(gt_str).get("gt_parse", {})
    except Exception: return {}
    out = {}; menu = parse.get("menu", [])
    if isinstance(menu, dict): menu = [menu]
    items = [{"name": str(m.get("nm","")), "price": str(m.get("price",""))}
             for m in menu if isinstance(m, dict) and "nm" in m]
    if items: out["items"] = items
    total = parse.get("total", {})
    if isinstance(total, dict) and "total_price" in total: out["total"] = str(total["total_price"])
    return out

INSTR = ("Extract the receipt into JSON with keys: items (a list of name and price) "
         "and total. Output only JSON.")

def infer(pil_img):
    FastVisionModel.for_inference(model)
    msg = [{"role":"user","content":[{"type":"image","image":pil_img},{"type":"text","text":INSTR}]}]
    inp = tokenizer.apply_chat_template(msg, add_generation_prompt=True, tokenize=True,
                                        return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**inp, max_new_tokens=256, do_sample=False)
    return tokenizer.batch_decode(out[:, inp["input_ids"].shape[1]:], skip_special_tokens=True)[0]

print("reloaded:", "model" in globals(), "| val_raw len:", len(val_raw))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2026.7.6: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

...

data/test-00000-of-00001-9c204eb3f4e1179(…):   0%|          | 0.00/234M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

reloaded: True | val_raw len: 60


In [6]:
print(infer(val_raw[0]["image"].convert("RGB")))

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"items": [{"name": "REAL GANACHE", "price": "16,500"}, {"name": "EGG TART", "price": "13,000"}, {"name": "PIZZA TOAST", "price": "16,000"}], "total": "45,500"}


In [5]:
import json, os, random, matplotlib.pyplot as plt
os.makedirs("/kaggle/working/samples", exist_ok=True)

# 1. save the metrics you already have
metrics = {"held_out_n": 60, "json_valid": 1.000, "total_exact": 0.900, "total_edit": 0.957}
json.dump(metrics, open("/kaggle/working/eval_metrics.json","w"), indent=2)

# 2. save 6 qualitative examples: receipt image + pred vs gold text
for i, ex in enumerate(random.sample(val_raw, 6)):
    gold = flatten_cord(ex["ground_truth"])
    if not gold: continue
    img = ex["image"].convert("RGB")
    if max(img.size) > 1024: img.thumbnail((1024,1024))
    pred_raw = infer(img)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 7), gridspec_kw={"width_ratios":[1,1]})
    a1.imshow(img); a1.axis("off"); a1.set_title("input receipt", fontsize=10)
    a2.axis("off")
    a2.text(0, 1, "PREDICTION:\n" + pred_raw[:400] + "\n\nGOLD total: " + str(gold.get("total","")),
            fontsize=8, va="top", family="monospace", wrap=True)
    fig.savefig(f"/kaggle/working/samples/sample_{i}.png", bbox_inches="tight", dpi=110)
    plt.close(fig)
print("saved eval_metrics.json and 6 sample images to /kaggle/working/samples/")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

saved eval_metrics.json and 6 sample images to /kaggle/working/samples/
